<a href="https://colab.research.google.com/github/medipallysreeya-gif/Machine-learning-skill/blob/main/Week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")   # comment out if running interactively
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------------------------------------------------------
# 1. Load Titanic dataset
# ---------------------------------------------------------
print("=" * 70)
print("TITANIC CLUSTERING: K-Means / Hierarchical / DBSCAN")
print("=" * 70)

df = pd.read_csv("Titanic-Dataset.csv")

# Drop irrelevant columns
drop_cols = ["PassengerId", "Name", "Ticket", "Cabin"]
df = df.drop(columns=drop_cols, errors="ignore")

# Numeric and categorical columns
NUMERIC_COLS = ["Age", "Fare", "SibSp", "Parch"]
CATEGORICAL_COLS = ["Sex", "Embarked", "Pclass"]

# Impute missing numeric values
imputer = SimpleImputer(strategy="median")
num_df = pd.DataFrame(imputer.fit_transform(df[NUMERIC_COLS]),
                      columns=NUMERIC_COLS, index=df.index)

# Encode categorical variables
cat_df = pd.get_dummies(df[CATEGORICAL_COLS], drop_first=True)

# Combine features
feature_df = pd.concat([num_df, cat_df], axis=1)
X = StandardScaler().fit_transform(feature_df)

print(f"Loaded {len(df):,} passengers, {feature_df.shape[1]} clustering features")

# ---------------------------------------------------------
# 2. Choose k for K-Means via elbow + silhouette
# ---------------------------------------------------------
k_range = range(2, 9)
sil_sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), min(500, len(X)), replace=False)
X_sil_sample = X[sil_sample_idx]

inertias, sil_scores = [], []
for k in k_range:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km_k.fit_predict(X)
    inertias.append(km_k.inertia_)
    sil_scores.append(silhouette_score(X_sil_sample, km_k.predict(X_sil_sample)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(list(k_range), inertias, marker="o")
ax1.set_title("Elbow method (inertia)")
ax1.set_xlabel("k")
ax1.set_ylabel("Inertia")

ax2.plot(list(k_range), sil_scores, marker="o", color="darkorange")
ax2.set_title("Silhouette score vs k")
ax2.set_xlabel("k")
ax2.set_ylabel("Silhouette score")

plt.tight_layout()
plt.savefig("titanic_kmeans_elbow_silhouette.png", dpi=150)
print("Saved plot -> titanic_kmeans_elbow_silhouette.png")
plt.close(fig)

best_k = list(k_range)[int(np.argmax(sil_scores))]
print(f"Silhouette-suggested k = {best_k}")

# ---------------------------------------------------------
# 3. K-Means on the FULL dataset
# ---------------------------------------------------------
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
km_labels = kmeans.fit_predict(X)

sil = silhouette_score(X[sil_sample_idx], km_labels[sil_sample_idx])
print(f"\nK-Means (k={best_k}) silhouette: {sil:.3f}")
print("Cluster sizes:", pd.Series(km_labels).value_counts().sort_index().to_dict())

# Profile clusters against survival
profiled = feature_df.copy()
profiled["KMeansCluster"] = km_labels
profiled["Survived"] = df["Survived"].values

summary = profiled.groupby("KMeansCluster").agg(
    n_passengers=("Survived", "size"),
    survival_rate=("Survived", "mean"),
    avg_age=("Age", "mean"),
    avg_fare=("Fare", "mean"),
    avg_sibsp=("SibSp", "mean"),
    avg_parch=("Parch", "mean"),
).round(3)

print("\n--- Cluster profiles (KMeansCluster) ---")
print(summary.to_string())
summary.to_csv("titanic_cluster_profiles.csv")
print("Saved table -> titanic_cluster_profiles.csv")

# ---------------------------------------------------------
# 4. Hierarchical + DBSCAN on a subsample
# ---------------------------------------------------------
sub_idx = np.random.RandomState(RANDOM_STATE).choice(len(X), min(500, len(X)), replace=False)
X_sub = X[sub_idx]

hc = AgglomerativeClustering(n_clusters=best_k, linkage="ward")
hc_labels = hc.fit_predict(X_sub)
sil_hc = silhouette_score(X_sub, hc_labels)
print(f"\nHierarchical (k={best_k}) silhouette: {sil_hc:.3f}")

db = DBSCAN(eps=4.0, min_samples=15)
db_labels = db.fit_predict(X_sub)
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = int(np.sum(db_labels == -1))
print(f"DBSCAN: {n_db_clusters} clusters, {n_noise} noise points")

if n_db_clusters >= 2:
    mask = db_labels != -1
    print(f"DBSCAN silhouette (excl. noise): {silhouette_score(X_sub[mask], db_labels[mask]):.3f}")

# Dendrogram
dendro_idx = np.random.choice(len(X_sub), size=min(80, len(X_sub)), replace=False)
Z = linkage(X_sub[dendro_idx], method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage)")
plt.xlabel("Passenger index")
plt.ylabel("Distance")
plt.tight_layout()
plt.savefig("titanic_dendrogram.png", dpi=150)
print("Saved plot -> titanic_dendrogram.png")
plt.close()

# ---------------------------------------------------------
# 5. PCA for visualization
# ---------------------------------------------------------
X_pca_full = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X)

# K-Means plot
fig, ax = plt.subplots(figsize=(6, 5.2))
ax.scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=km_labels, cmap="tab10", s=8, alpha=0.6)
ax.set_title(f"K-Means, full data ({best_k} clusters)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.savefig("titanic_kmeans_pca.png", dpi=150)
print("Saved plot -> titanic_kmeans_pca.png")
plt.close(fig)

# Hierarchical + DBSCAN plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2))
coords_sub = X_pca_full[sub_idx]

axes[0].scatter(coords_sub[:, 0], coords_sub[:, 1], c=hc_labels, cmap="tab10", s=8, alpha=0.6)
axes[0].set_title(f"Hierarchical, subsample ({best_k} clusters)")

noise_mask = db_labels == -1
axes[1].scatter(coords_sub[~noise_mask, 0], coords_sub[~noise_mask, 1],
                c=db_labels[~noise_mask], cmap="tab10", s=8, alpha=0.6)
axes[1].scatter(coords_sub[noise_mask, 0], coords_sub[noise_mask, 1],
                c="lightgray", s=8, alpha=0.5, marker="x", label="noise")
axes[1].legend(loc="upper right", fontsize=8)
axes[1].set_title(f"DBSCAN, subsample ({n_db_clusters} clusters)")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
plt.savefig("titanic_hierarchical_dbscan_pca.png", dpi=150)
print("Saved plot -> titanic_hierarchical_dbscan_pca.png")
plt.close(fig)

# Save clustered dataset
df_out = df.copy()
df_out["KMeansCluster"] = km_labels
df_out.to_csv("titanic_with_clusters.csv", index=False)
print("\nSaved deliverable -> titanic_with_clusters.csv (original data + KMeansCluster column)")

print("\nDone. K-Means gives a clean segmentation of passengers into profiles; "
      "the survival rate per cluster (see titanic_cluster_profiles.csv) shows which groups had higher chances of survival.")


TITANIC CLUSTERING: K-Means / Hierarchical / DBSCAN
Loaded 891 passengers, 8 clustering features
Saved plot -> titanic_kmeans_elbow_silhouette.png
Silhouette-suggested k = 2

K-Means (k=2) silhouette: 0.373
Cluster sizes: {0: 814, 1: 77}

--- Cluster profiles (KMeansCluster) ---
               n_passengers  survival_rate  avg_age  avg_fare  avg_sibsp  avg_parch
KMeansCluster                                                                      
0                       814          0.383   29.487    33.995      0.532      0.402
1                        77          0.390   28.032    13.276      0.429      0.169
Saved table -> titanic_cluster_profiles.csv

Hierarchical (k=2) silhouette: 0.366
DBSCAN: 1 clusters, 0 noise points
Saved plot -> titanic_dendrogram.png
Saved plot -> titanic_kmeans_pca.png
Saved plot -> titanic_hierarchical_dbscan_pca.png

Saved deliverable -> titanic_with_clusters.csv (original data + KMeansCluster column)

Done. K-Means gives a clean segmentation of passengers 